In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
Accused = pd.read_csv('Accused.csv')
act = pd.read_csv('Act_Section.csv')
Chargesheet = pd.read_csv('Chargesheet.csv')
PS_Unit = pd.read_csv('PS_Unit.csv')
Victim = pd.read_csv('Victim.csv')
Dispose = pd.read_csv('Dispose.csv')
Crime = pd.read_csv('Crime.csv')

print("Accused shape:", Accused.shape)
print("Act shape:", act.shape)
print("Chargesheet shape:", Chargesheet.shape)
print("PS_Unit shape:", PS_Unit.shape)
print("Victim shape:", Victim.shape)
print("Dispose:", Dispose.shape)
print("Crime:", Crime.shape)

Accused shape: (237773, 15)
Act shape: (349029, 6)
Chargesheet shape: (54325, 6)
PS_Unit shape: (1856, 4)
Victim shape: (141398, 15)
Dispose: (17660, 5)
Crime: (211863, 20)


In [3]:
print("Crime columns:", Crime.columns.tolist())
print("Dispose columns:", Dispose.columns.tolist())
print("Act columns:", act.columns.tolist())
print("PS_Unit columns:", PS_Unit.columns.tolist())

Crime columns: ['District_Name', 'UnitName', 'FIRNo', 'Pin_Code', 'Direction_Distance', 'Beat', 'Village', 'FIR Type', 'FIR_Stage', 'CrimeGroup_Name', 'CrimeHead_Name', 'Latitude', 'Longitude', 'Fir_id', 'Unit_ID', 'Offence_From_Date', 'Offence_To_Date', 'FIR_Date', 'Unnamed: 18', 'Unnamed: 19']
Dispose columns: ['CaseMasterID', 'CASEChargesheetID', 'Fir_id', 'Unit_id', 'Dispose_Date']
Act columns: ['Act_Code', 'Act_Description', 'Section_Code', 'Sec_Description', 'Fir_id', 'Unit_id']
PS_Unit columns: ['UnitID', 'UnitName', 'Latitude', 'Longitude']


In [4]:
# Drop junk columns
Crime.drop(columns=['Unnamed: 18', 'Unnamed: 19'], inplace=True, errors='ignore')

# Standardize join key column name
Crime.rename(columns={'Unit_ID': 'Unit_id'}, inplace=True)

# Convert dates
Crime['FIR_Date'] = pd.to_datetime(Crime['FIR_Date'], errors='coerce')
Dispose['Dispose_Date'] = pd.to_datetime(Dispose['Dispose_Date'], errors='coerce')

# Extract date parts
Crime['FIR_Year']       = Crime['FIR_Date'].dt.year
Crime['FIR_Month']      = Crime['FIR_Date'].dt.month
Crime['FIR_Month_Name'] = Crime['FIR_Date'].dt.strftime('%B')

print("Crime cleaned:", Crime.shape)
print("Columns:", Crime.columns.tolist())

Crime cleaned: (211863, 21)
Columns: ['District_Name', 'UnitName', 'FIRNo', 'Pin_Code', 'Direction_Distance', 'Beat', 'Village', 'FIR Type', 'FIR_Stage', 'CrimeGroup_Name', 'CrimeHead_Name', 'Latitude', 'Longitude', 'Fir_id', 'Unit_id', 'Offence_From_Date', 'Offence_To_Date', 'FIR_Date', 'FIR_Year', 'FIR_Month', 'FIR_Month_Name']


In [5]:
# Create composite key on all three tables
Crime['Key']   = Crime['Fir_id']   + '-' + Crime['Unit_id']
Dispose['Key'] = Dispose['Fir_id'] + '-' + Dispose['Unit_id']
act['Key']     = act['Fir_id']     + '-' + act['Unit_id']

# Deduplicate crime to one row per FIR
crime_unique = Crime.drop_duplicates(subset=['Key']).copy()

print("Unique FIRs:", len(crime_unique))
print("Crime keys sample:", crime_unique['Key'].head(3).tolist())

Unique FIRs: 140585
Crime keys sample: ['FIR0001-UNIT001', 'FIR0009-UNIT001', 'FIR0022-UNIT001']


In [6]:
# Convert dates
crime_unique['FIR_Date'] = pd.to_datetime(crime_unique['FIR_Date'], errors='coerce')
Dispose['Dispose_Date'] = pd.to_datetime(Dispose['Dispose_Date'], errors='coerce')

# Check
print("FIR_Date dtype:", crime_unique['FIR_Date'].dtype)
print("Dispose_Date dtype:", Dispose['Dispose_Date'].dtype)
print("FIR_Date nulls:", crime_unique['FIR_Date'].isnull().sum())
print("Sample dates:", crime_unique['FIR_Date'].head(3).tolist())

FIR_Date dtype: datetime64[ns]
Dispose_Date dtype: datetime64[ns]
FIR_Date nulls: 0
Sample dates: [Timestamp('2021-08-08 10:45:00'), Timestamp('2021-08-20 10:00:00'), Timestamp('2021-09-01 12:00:00')]


In [7]:
# Step 2 — Join Dispose table to flag Disposed vs Pending
crime_unique = crime_unique.merge(
    Dispose[['Key', 'Dispose_Date']],
    on='Key',
    how='left'
)

# Create flags
crime_unique['Is_Disposed'] = crime_unique['Dispose_Date'].notnull().astype(int)
crime_unique['Is_Pending']  = (crime_unique['Dispose_Date'].isnull()).astype(int)

# Check
print("Total FIRs:", len(crime_unique))
print("Disposed:", crime_unique['Is_Disposed'].sum())
print("Pending:", crime_unique['Is_Pending'].sum())

Total FIRs: 140689
Disposed: 6125
Pending: 134564


In [8]:
act['Key'] = act['Fir_id'] + '-' + act['Unit_id']
print(act['Key'].head(3).tolist())

['FIR13871-UNIT016', 'FIR0039-UNIT016', 'FIR13448-UNIT016']


In [9]:
Crime.isnull().sum()
Dispose.isnull().sum()
act.isnull().sum()

Act_Code           0
Act_Description    0
Section_Code       0
Sec_Description    0
Fir_id             0
Unit_id            0
Key                0
dtype: int64

In [10]:
Crime.duplicated().sum()

26256

In [11]:
Crime['Key'].nunique()
len(Crime)

211863

In [12]:
Crime['FIR_Month'].unique()

array([ 8,  9, 10, 11, 12,  1,  2,  3,  4,  5,  6,  7])

In [13]:
merged = crime_unique.merge(Dispose[['Key','Dispose_Date']], on='Key', how='left')
merged = merged.merge(act[['Key','Act_Description']], on='Key', how='left')
merged

,District_Name,UnitName,FIRNo,Pin_Code,Direction_Distance,Beat,Village,FIR Type,FIR_Stage,CrimeGroup_Name,...,FIR_Date,FIR_Year,FIR_Month,FIR_Month_Name,Key,Dispose_Date_x,Is_Disposed,Is_Pending,Dispose_Date_y,Act_Description
0,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Non-Heinous,Disposal,CYBER CRIME,...,2021-08-08 10:45:00,2021,8,August,FIR0001-UNIT001,NaT,0,1,NaT,INFORMATION TECHNOLOGY ACT 2000
1,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Non-Heinous,Disposal,CYBER CRIME,...,2021-08-08 10:45:00,2021,8,August,FIR0001-UNIT001,NaT,0,1,NaT,INFORMATION TECHNOLOGY ACT 2000
2,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Non-Heinous,Pending,CYBER CRIME,...,2021-08-20 10:00:00,2021,8,August,FIR0009-UNIT001,NaT,0,1,NaT,INFORMATION TECHNOLOGY ACT 2000
3,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Heinous,Disposal,CYBER CRIME,...,2021-09-01 12:00:00,2021,9,September,FIR0022-UNIT001,NaT,0,1,NaT,INFORMATION TECHNOLOGY ACT 2000
4,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Heinous,Disposal,CYBER CRIME,...,2021-09-01 12:00:00,2021,9,September,FIR0022-UNIT001,NaT,0,1,NaT,INFORMATION TECHNOLOGY ACT 2000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
351678,Bengaluru City,Kadugodi PS,1.044300e+17,0,4,2010000008,SIGEHALLY,Non-Heinous,Traced,MISSING PERSON,...,2024-12-21 13:00:00,2024,12,December,FIR140473-UNIT050,NaT,0,1,NaT,"THE BHARATIYA NYAYA SANHITA (BNS), 2023"
351679,Bengaluru City,Kadugodi PS,1.044300e+17,0,49,2010000007,CHINNAGENAHALLY,Non-Heinous,Traced,MISSING PERSON,...,2025-01-03 11:00:00,2025,1,January,FIR140494-UNIT050,NaT,0,1,NaT,"THE BHARATIYA NYAYA SANHITA (BNS), 2023"
351680,Bengaluru City,Kadugodi PS,1.044300e+17,0,2,2010000003,CHANNASANDRA,Non-Heinous,Traced,MISSING PERSON,...,2025-01-03 21:00:00,2025,1,January,FIR140499-UNIT050,NaT,0,1,NaT,"THE BHARATIYA NYAYA SANHITA (BNS), 2023"
351681,Bengaluru City,Kadugodi PS,1.044300e+17,0,3,2010000003,CHANNASANDRA,Non-Heinous,Traced,MISSING PERSON,...,2025-01-02 21:00:00,2025,1,January,FIR140509-UNIT050,NaT,0,1,NaT,"THE BHARATIYA NYAYA SANHITA (BNS), 2023"


In [14]:
act_grouped = act.groupby('Key')['Act_Description'].apply(lambda x: ', '.join(x)).reset_index()

merged = crime_unique.merge(Dispose[['Key','Dispose_Date']], on='Key', how='left')
merged = merged.merge(act_grouped, on='Key', how='left')
merged

,District_Name,UnitName,FIRNo,Pin_Code,Direction_Distance,Beat,Village,FIR Type,FIR_Stage,CrimeGroup_Name,...,FIR_Date,FIR_Year,FIR_Month,FIR_Month_Name,Key,Dispose_Date_x,Is_Disposed,Is_Pending,Dispose_Date_y,Act_Description
0,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Non-Heinous,Disposal,CYBER CRIME,...,2021-08-08 10:45:00,2021,8,August,FIR0001-UNIT001,NaT,0,1,NaT,"INFORMATION TECHNOLOGY ACT 2000, INFORMATION ..."
1,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Non-Heinous,Pending,CYBER CRIME,...,2021-08-20 10:00:00,2021,8,August,FIR0009-UNIT001,NaT,0,1,NaT,INFORMATION TECHNOLOGY ACT 2000
2,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Heinous,Disposal,CYBER CRIME,...,2021-09-01 12:00:00,2021,9,September,FIR0022-UNIT001,NaT,0,1,NaT,"INFORMATION TECHNOLOGY ACT 2000, INFORMATION ..."
3,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Non-Heinous,Disposal,CYBER CRIME,...,2021-09-09 17:00:00,2021,9,September,FIR0029-UNIT001,NaT,0,1,NaT,"INFORMATION TECHNOLOGY ACT 2000, INFORMATION ..."
4,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Non-Heinous,Disposal,CYBER CRIME,...,2021-09-18 13:00:00,2021,9,September,FIR0041-UNIT001,NaT,0,1,NaT,"INFORMATION TECHNOLOGY ACT 2000, INFORMATION ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140898,Bengaluru City,Kadugodi PS,1.044300e+17,0,4,2010000008,SIGEHALLY,Non-Heinous,Traced,MISSING PERSON,...,2024-12-21 13:00:00,2024,12,December,FIR140473-UNIT050,NaT,0,1,NaT,"THE BHARATIYA NYAYA SANHITA (BNS), 2023"
140899,Bengaluru City,Kadugodi PS,1.044300e+17,0,49,2010000007,CHINNAGENAHALLY,Non-Heinous,Traced,MISSING PERSON,...,2025-01-03 11:00:00,2025,1,January,FIR140494-UNIT050,NaT,0,1,NaT,"THE BHARATIYA NYAYA SANHITA (BNS), 2023"
140900,Bengaluru City,Kadugodi PS,1.044300e+17,0,2,2010000003,CHANNASANDRA,Non-Heinous,Traced,MISSING PERSON,...,2025-01-03 21:00:00,2025,1,January,FIR140499-UNIT050,NaT,0,1,NaT,"THE BHARATIYA NYAYA SANHITA (BNS), 2023"
140901,Bengaluru City,Kadugodi PS,1.044300e+17,0,3,2010000003,CHANNASANDRA,Non-Heinous,Traced,MISSING PERSON,...,2025-01-02 21:00:00,2025,1,January,FIR140509-UNIT050,NaT,0,1,NaT,"THE BHARATIYA NYAYA SANHITA (BNS), 2023"


In [15]:
import calendar
merged['FIR_Month_Name'] = merged['FIR_Month'].apply(lambda x: calendar.month_name[x])
merged

,District_Name,UnitName,FIRNo,Pin_Code,Direction_Distance,Beat,Village,FIR Type,FIR_Stage,CrimeGroup_Name,...,FIR_Date,FIR_Year,FIR_Month,FIR_Month_Name,Key,Dispose_Date_x,Is_Disposed,Is_Pending,Dispose_Date_y,Act_Description
0,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Non-Heinous,Disposal,CYBER CRIME,...,2021-08-08 10:45:00,2021,8,August,FIR0001-UNIT001,NaT,0,1,NaT,"INFORMATION TECHNOLOGY ACT 2000, INFORMATION ..."
1,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Non-Heinous,Pending,CYBER CRIME,...,2021-08-20 10:00:00,2021,8,August,FIR0009-UNIT001,NaT,0,1,NaT,INFORMATION TECHNOLOGY ACT 2000
2,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Heinous,Disposal,CYBER CRIME,...,2021-09-01 12:00:00,2021,9,September,FIR0022-UNIT001,NaT,0,1,NaT,"INFORMATION TECHNOLOGY ACT 2000, INFORMATION ..."
3,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Non-Heinous,Disposal,CYBER CRIME,...,2021-09-09 17:00:00,2021,9,September,FIR0029-UNIT001,NaT,0,1,NaT,"INFORMATION TECHNOLOGY ACT 2000, INFORMATION ..."
4,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Non-Heinous,Disposal,CYBER CRIME,...,2021-09-18 13:00:00,2021,9,September,FIR0041-UNIT001,NaT,0,1,NaT,"INFORMATION TECHNOLOGY ACT 2000, INFORMATION ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140898,Bengaluru City,Kadugodi PS,1.044300e+17,0,4,2010000008,SIGEHALLY,Non-Heinous,Traced,MISSING PERSON,...,2024-12-21 13:00:00,2024,12,December,FIR140473-UNIT050,NaT,0,1,NaT,"THE BHARATIYA NYAYA SANHITA (BNS), 2023"
140899,Bengaluru City,Kadugodi PS,1.044300e+17,0,49,2010000007,CHINNAGENAHALLY,Non-Heinous,Traced,MISSING PERSON,...,2025-01-03 11:00:00,2025,1,January,FIR140494-UNIT050,NaT,0,1,NaT,"THE BHARATIYA NYAYA SANHITA (BNS), 2023"
140900,Bengaluru City,Kadugodi PS,1.044300e+17,0,2,2010000003,CHANNASANDRA,Non-Heinous,Traced,MISSING PERSON,...,2025-01-03 21:00:00,2025,1,January,FIR140499-UNIT050,NaT,0,1,NaT,"THE BHARATIYA NYAYA SANHITA (BNS), 2023"
140901,Bengaluru City,Kadugodi PS,1.044300e+17,0,3,2010000003,CHANNASANDRA,Non-Heinous,Traced,MISSING PERSON,...,2025-01-02 21:00:00,2025,1,January,FIR140509-UNIT050,NaT,0,1,NaT,"THE BHARATIYA NYAYA SANHITA (BNS), 2023"


In [16]:
summary = merged.groupby(
    ['Unit_id', 'FIR_Year', 'FIR_Month_Name', 'Act_Description']
).agg(
    Total_Cases=('Key', 'count'),
    Disposed_Cases=('Is_Disposed', 'sum'),
    Pending_Cases=('Is_Pending', 'sum')
).reset_index()
summary['Pending_Percentage'] = (
    summary['Pending_Cases'] / summary['Total_Cases'] * 100
).round(2)
summary

,Unit_id,FIR_Year,FIR_Month_Name,Act_Description,Total_Cases,Disposed_Cases,Pending_Cases,Pending_Percentage
0,UNIT001,2020,April,INFORMATION TECHNOLOGY ACT 2000,5,0,5,100.0
1,UNIT001,2020,April,"INFORMATION TECHNOLOGY ACT 2000, INFORMATION ...",1,0,1,100.0
2,UNIT001,2020,April,"IPC 1860, INFORMATION TECHNOLOGY ACT 2000",4,0,4,100.0
3,UNIT001,2020,April,"IPC 1860, INFORMATION TECHNOLOGY ACT 2000, IN...",99,0,99,100.0
4,UNIT001,2020,April,"IPC 1860, IPC 1860, INFORMATION TECHNOLOGY AC...",16,0,16,100.0
...,...,...,...,...,...,...,...,...
28987,UNIT050,2024,September,"THE BHARATIYA NYAYA SANHITA (BNS), 2023, THE B...",1,0,1,100.0
28988,UNIT050,2025,January,"INFORMATION TECHNOLOGY ACT 2000, INFORMATION ...",2,0,2,100.0
28989,UNIT050,2025,January,"INFORMATION TECHNOLOGY ACT 2008, INFORMATION T...",2,0,2,100.0
28990,UNIT050,2025,January,"THE BHARATIYA NYAYA SANHITA (BNS), 2023",6,0,6,100.0


In [ ]:
summary.to_csv('crime_summary.csv', index=False)

In [17]:
# 1. Missing values
act['Act_Description'] = act['Act_Description'].fillna('Unknown')

# 2. Remove exact duplicates
Crime = Crime.drop_duplicates()
Dispose = Dispose.drop_duplicates()
act = act.drop_duplicates()

# 3. Check month validity
Crime = Crime[(Crime['FIR_Month'] >= 1) & (Crime['FIR_Month'] <= 12)]
Crime

,District_Name,UnitName,FIRNo,Pin_Code,Direction_Distance,Beat,Village,FIR Type,FIR_Stage,CrimeGroup_Name,...,Longitude,Fir_id,Unit_id,Offence_From_Date,Offence_To_Date,FIR_Date,FIR_Year,FIR_Month,FIR_Month_Name,Key
0,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Non-Heinous,Disposal,CYBER CRIME,...,0.000000,FIR0001,UNIT001,2021/06/26 14:20:00,2021/06/26 22:00:00,2021-08-08 10:45:00,2021,8,August,FIR0001-UNIT001
3,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Non-Heinous,Pending,CYBER CRIME,...,0.000000,FIR0009,UNIT001,2021/07/11 0:00:00,2021/07/11 0:01:00,2021-08-20 10:00:00,2021,8,August,FIR0009-UNIT001
4,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Non-Heinous,Under Investigation,CYBER CRIME,...,0.000000,FIR0009,UNIT001,2021/07/11 0:00:00,2021/07/11 0:01:00,2021-08-20 10:00:00,2021,8,August,FIR0009-UNIT001
5,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Non-Heinous,Under Inquiry,CYBER CRIME,...,0.000000,FIR0009,UNIT001,2021/07/11 0:00:00,2021/07/11 0:01:00,2021-08-20 10:00:00,2021,8,August,FIR0009-UNIT001
6,Bengaluru City,North CEN Crime PS,1.044320e+17,0,0,2019000001,YPR CENPS,Heinous,Disposal,CYBER CRIME,...,0.000000,FIR0022,UNIT001,2021/07/23 0:00:00,2021/07/29 0:00:00,2021-09-01 12:00:00,2021,9,September,FIR0022-UNIT001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211858,Bengaluru City,Kadugodi PS,1.044300e+17,0,4,2010000008,SIGEHALLY,Non-Heinous,Traced,MISSING PERSON,...,77.444351,FIR140473,UNIT050,2024/12/20 9:00:00,2024/12/21 12:30:00,2024-12-21 13:00:00,2024,12,December,FIR140473-UNIT050
211859,Bengaluru City,Kadugodi PS,1.044300e+17,0,49,2010000007,CHINNAGENAHALLY,Non-Heinous,Traced,MISSING PERSON,...,77.748017,FIR140494,UNIT050,2025/01/02 13:00:00,2025/01/02 14:30:00,2025-01-03 11:00:00,2025,1,January,FIR140494-UNIT050
211860,Bengaluru City,Kadugodi PS,1.044300e+17,0,2,2010000003,CHANNASANDRA,Non-Heinous,Traced,MISSING PERSON,...,77.757632,FIR140499,UNIT050,2025/01/02 19:00:00,2025/01/03 20:00:00,2025-01-03 21:00:00,2025,1,January,FIR140499-UNIT050
211861,Bengaluru City,Kadugodi PS,1.044300e+17,0,3,2010000003,CHANNASANDRA,Non-Heinous,Traced,MISSING PERSON,...,77.766844,FIR140509,UNIT050,2025/01/02 11:00:00,2025/01/02 20:00:00,2025-01-02 21:00:00,2025,1,January,FIR140509-UNIT050


In [24]:
summary = merged.groupby(
    ['Unit_id', 'FIR_Year', 'FIR_Month_Name', 'Act_Description']
).agg(
    Total_Cases=('Key', 'count'),
    Disposed_Cases=('Is_Disposed', 'sum'),
    Pending_Cases=('Is_Pending', 'sum')
).reset_index()

summary['Pending_Percentage'] = (
    summary['Pending_Cases'] / summary['Total_Cases'] * 100
).round(2)
summary

,Unit_id,FIR_Year,FIR_Month_Name,Act_Description,Total_Cases,Disposed_Cases,Pending_Cases,Pending_Percentage
0,UNIT001,2020,April,INFORMATION TECHNOLOGY ACT 2000,5,0,5,100.0
1,UNIT001,2020,April,"INFORMATION TECHNOLOGY ACT 2000, INFORMATION ...",1,0,1,100.0
2,UNIT001,2020,April,"IPC 1860, INFORMATION TECHNOLOGY ACT 2000",4,0,4,100.0
3,UNIT001,2020,April,"IPC 1860, INFORMATION TECHNOLOGY ACT 2000, IN...",99,0,99,100.0
4,UNIT001,2020,April,"IPC 1860, IPC 1860, INFORMATION TECHNOLOGY AC...",16,0,16,100.0
...,...,...,...,...,...,...,...,...
28987,UNIT050,2024,September,"THE BHARATIYA NYAYA SANHITA (BNS), 2023, THE B...",1,0,1,100.0
28988,UNIT050,2025,January,"INFORMATION TECHNOLOGY ACT 2000, INFORMATION ...",2,0,2,100.0
28989,UNIT050,2025,January,"INFORMATION TECHNOLOGY ACT 2008, INFORMATION T...",2,0,2,100.0
28990,UNIT050,2025,January,"THE BHARATIYA NYAYA SANHITA (BNS), 2023",6,0,6,100.0


In [ ]:
summary.to_csv('crime_summary_F.csv', index=False)

In [19]:
print(summary.head())
print(summary.shape)
print(summary['Total_Cases'].sum())

   Unit_id  FIR_Year FIR_Month_Name  \
0  UNIT001      2020          April   
1  UNIT001      2020          April   
2  UNIT001      2020          April   
3  UNIT001      2020          April   
4  UNIT001      2020          April   

                                     Act_Description  Total_Cases  \
0                   INFORMATION TECHNOLOGY  ACT 2000            5   
1  INFORMATION TECHNOLOGY  ACT 2000, INFORMATION ...            1   
2         IPC 1860, INFORMATION TECHNOLOGY  ACT 2000            4   
3  IPC 1860, INFORMATION TECHNOLOGY  ACT 2000, IN...           99   
4  IPC 1860, IPC 1860, INFORMATION TECHNOLOGY  AC...           16   

   Disposed_Cases  Pending_Cases  Pending_Percentage  
0               0              5               100.0  
1               0              1               100.0  
2               0              4               100.0  
3               0             99               100.0  
4               0             16               100.0  
(28992, 8)
139089


In [17]:
# Final clean aggregation with UnitName
summary_final = merged.groupby(
    ['UnitName', 'FIR_Year', 'FIR_Month', 'FIR_Month_Name', 'Act_Description']
).agg(
    Total_Cases=('Key', 'count'),
    Disposed_Cases=('Is_Disposed', 'sum'),
    Pending_Cases=('Is_Pending', 'sum')
).reset_index()

summary_final['Pending_Percentage'] = (
    summary_final['Pending_Cases'] / summary_final['Total_Cases'] * 100
).round(2)

# Sort by year and month for clean ordering
summary_final = summary_final.sort_values(['FIR_Year', 'FIR_Month'])

# Save
summary_final.to_csv('cru_aggregated.csv', index=False)

print("Shape:", summary_final.shape)
print("Total cases:", summary_final['Total_Cases'].sum())
print("\nSample:")
print(summary_final.head())

Shape: (28992, 9)
Total cases: 139089

Sample:
           UnitName  FIR_Year  FIR_Month FIR_Month_Name  \
611   Ashoknagar PS      2020          1        January   
612   Ashoknagar PS      2020          1        January   
613   Ashoknagar PS      2020          1        January   
1925   Banaswadi PS      2020          1        January   
1926   Banaswadi PS      2020          1        January   

                                     Act_Description  Total_Cases  \
611                                         IPC 1860            1   
612                     IPC 1860, IPC 1860, IPC 1860            2   
613   IPC 1860, IPC 1860, KARNATAKA POLICE ACT, 1963            1   
1925                                        IPC 1860            1   
1926          IPC 1860, IPC 1860, IPC 1860, IPC 1860            1   

      Disposed_Cases  Pending_Cases  Pending_Percentage  
611                0              1               100.0  
612                0              2               100.0  
613      

In [20]:
print(crime_unique.columns.tolist())

['District_Name', 'UnitName', 'FIRNo', 'Pin_Code', 'Direction_Distance', 'Beat', 'Village', 'FIR Type', 'FIR_Stage', 'CrimeGroup_Name', 'CrimeHead_Name', 'Latitude', 'Longitude', 'Fir_id', 'Unit_id', 'Offence_From_Date', 'Offence_To_Date', 'FIR_Date', 'FIR_Year', 'FIR_Month', 'FIR_Month_Name', 'Key', 'Dispose_Date', 'Is_Disposed', 'Is_Pending']


In [21]:
# Take only the PRIMARY act per FIR (first one)
act_primary = act.drop_duplicates(subset=['Key'])[['Key', 'Act_Description']]

# Merge only act (Dispose already in crime_unique)
merged_clean = crime_unique.merge(
    act_primary,
    on='Key',
    how='left'
)

merged_clean['Act_Description'] = merged_clean['Act_Description'].fillna('Unknown')

# Final aggregation
summary_final = merged_clean.groupby(
    ['UnitName', 'FIR_Year', 'FIR_Month', 'FIR_Month_Name', 'Act_Description']
).agg(
    Total_Cases=('Key', 'count'),
    Disposed_Cases=('Is_Disposed', 'sum'),
    Pending_Cases=('Is_Pending', 'sum')
).reset_index()

summary_final['Pending_Percentage'] = (
    summary_final['Pending_Cases'] / summary_final['Total_Cases'] * 100
).round(2)

summary_final = summary_final.sort_values(['FIR_Year', 'FIR_Month']).reset_index(drop=True)

summary_final.to_csv('cru_aggregated.csv', index=False)

print("Shape:", summary_final.shape)
print("\nUnique Act types:", summary_final['Act_Description'].nunique())
print("\nTop act types:")
print(summary_final['Act_Description'].value_counts().head(8))
print("\nSample:")
print(summary_final.head())

Shape: (9453, 9)

Unique Act types: 101

Top act types:
Act_Description
IPC 1860                                                2897
NARCOTIC DRUGS AND PSYCHOTROPIC SUBSTANCES ACT, 1985    1037
KARNATAKA POLICE ACT, 1963                               823
NARCOTIC DRUGS & PSYCHOTROPIC SUBSTANCES ACT, 1985       820
INFORMATION TECHNOLOGY  ACT 2000                         761
INFORMATION TECHNOLOGY ACT 2008                          470
ARMS ACT, 1959                                           414
THE BHARATIYA NYAYA SANHITA (BNS), 2023                  260
Name: count, dtype: int64

Sample:
                UnitName  FIR_Year  FIR_Month FIR_Month_Name Act_Description  \
0          Ashoknagar PS      2020          1        January        IPC 1860   
1           Banaswadi PS      2020          1        January        IPC 1860   
2          Bellanduru PS      2020          1        January        IPC 1860   
3              H.A.L. PS      2020          1        January        IPC 1860   
4  Je

In [22]:
# Clean act name inconsistencies
summary_final['Act_Description'] = summary_final['Act_Description'].str.strip()
summary_final['Act_Description'] = summary_final['Act_Description'].replace({
    'NARCOTIC DRUGS & PSYCHOTROPIC SUBSTANCES ACT, 1985': 'NARCOTIC DRUGS AND PSYCHOTROPIC SUBSTANCES ACT, 1985',
    'INFORMATION TECHNOLOGY  ACT 2000': 'INFORMATION TECHNOLOGY ACT 2000',
})

summary_final.to_csv('cru_aggregated.csv', index=False)

print("Unique Act types after cleaning:", summary_final['Act_Description'].nunique())
print(summary_final['Act_Description'].value_counts().head(8))

Unique Act types after cleaning: 100
Act_Description
IPC 1860                                                  2897
NARCOTIC DRUGS AND PSYCHOTROPIC SUBSTANCES ACT, 1985      1857
KARNATAKA POLICE ACT, 1963                                 823
INFORMATION TECHNOLOGY ACT 2000                            761
INFORMATION TECHNOLOGY ACT 2008                            470
ARMS ACT, 1959                                             414
THE BHARATIYA NYAYA SANHITA (BNS), 2023                    260
COTPA, Cigarettes and Other Tobacco Products,Act 2003,     233
Name: count, dtype: int64
